# Notebook 1: Hyperspectral Data Exploration & EDA
**Course:** INT396 - Unsupervised Learning
**Project:** PCA and UMAP-Driven Dimensionality Reduction of Hyperspectral Crop Data for Disease-Relevant Feature Preservation

This notebook loads the real potato disease hyperspectral dataset (Zenodo 14267877) and conducts exploratory data analysis:
- Data structure & shapes
- Data quality verification
- Class distribution
- Mean spectral reflectance signatures by disease category

In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_dataset
from src.data_validation import validate_hyperspectral_data
from src.eda import (
    calculate_feature_statistics,
    calculate_class_distribution,
    compute_spectral_signatures_by_class,
    compute_band_variance_ranking
)

# Load Day 70 symptomatic potato disease dataset
X, y, meta, bands, wls = load_dataset('potato_dai70_vnir')
print(f"Loaded X shape: {X.shape}, y shape: {y.shape if y is not None else None}")
print(f"Spectral range: {wls[0]:.1f} nm to {wls[-1]:.1f} nm across {len(bands)} bands")

In [ ]:
# 1. Data Integrity Validation
val_report = validate_hyperspectral_data(X, y)
print("Validation passed:", val_report['is_valid'])
print("Statistics:", val_report['statistics'])

In [ ]:
# 2. Disease Class Distribution
class_df = calculate_class_distribution(y)
print("Class breakdown:
", class_df)

In [ ]:
# 3. Plot Mean Spectral Reflectance Signatures by Class
signatures = compute_spectral_signatures_by_class(X, y, wls)

fig, ax = plt.subplots(figsize=(10, 4.5), facecolor='white')
colors = sns.color_palette("tab10", len(signatures))

for (cls_name, sig), col in zip(signatures.items(), colors):
    mean_curve = sig['mean']
    std_curve = sig['std']
    wl_arr = sig['wavelengths']
    ax.plot(wl_arr, mean_curve, label=cls_name, color=col, linewidth=2)
    ax.fill_between(wl_arr, mean_curve - std_curve, mean_curve + std_curve, color=col, alpha=0.15)

ax.set_xlabel("Wavelength (nm)", fontweight='bold')
ax.set_ylabel("Spectral Reflectance", fontweight='bold')
ax.set_title("Mean Spectral Reflectance Signatures (±1 Std Dev Ribbon)", fontweight='bold')
ax.legend(loc='upper left')
ax.grid(True, linestyle='--', alpha=0.4)
plt.show()